# Notebook 16.1  A miniature end-to-end Arabic speech project

Companion to **Chapter 16, End-to-End Arabic Speech Projects**.

This notebook runs the whole project lifecycle in miniature: a small Arabic command
set with metadata, Arabic text normalization, **speaker-disjoint** train/dev/test splits
with a leakage check, log-mel features (built with NumPy, tying back to Chapter 3),
SpecAugment, a baseline classifier evaluated with command accuracy and an **out-of-set
false-accept rate**, and a filled **datasheet**.

To keep it runnable anywhere with no downloads, the audio is **synthetic and
deterministic**: the numbers below only check that the pipeline is correct, not real
performance. The final section shows how to swap in real **Common Voice Arabic** clips.

Dependencies: only `numpy` and `scikit-learn` (both preinstalled in Google Colab).


In [ ]:
import numpy as np, re
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(0)
SR = 16000  # sampling rate (Hz), 16 kHz as in Chapter 3

## 1. A small Arabic command dataset with metadata

A real project records consented Gulf-dialect speakers (Section 16.3). Here we synthesize
a tiny dataset so the notebook runs offline. Each clip carries the metadata a fair split
and a datasheet need: speaker, gender, dialect, the Arabic command text, and whether it is
an in-vocabulary command or **out-of-set** (OOS) speech.


In [ ]:
COMMANDS = {'تشغيل':'turn_on', 'إيقاف':'turn_off', 'رفع':'up', 'خفض':'down'}  # Arabic -> label
CMD_LIST = list(COMMANDS)
SPEAKERS = [f'spk{i:02d}' for i in range(20)]
GENDER   = {s:('f' if i%2 else 'm') for i,s in enumerate(SPEAKERS)}
DIALECT  = {s:['najdi','hijazi','khaleeji'][i%3] for i,s in enumerate(SPEAKERS)}

def synth_clip(label_idx, spk_idx, is_command=True):
    dur = 0.6; t = np.linspace(0, dur, int(SR*dur), endpoint=False)
    if is_command:                       # label-specific tones + speaker formant offset
        base = 180 + 40*label_idx
        form = 600 + 150*label_idx + 30*(spk_idx % 5)
        sig = 0.5*np.sin(2*np.pi*base*t) + 0.3*np.sin(2*np.pi*form*t)
    else:                                # out-of-set 'garbage' speech
        sig = 0.4*rng.standard_normal(t.size)
    sig = sig + 0.05*rng.standard_normal(t.size)   # noisy living-room channel
    return sig.astype(np.float32)

rows, waves = [], []
for si, spk in enumerate(SPEAKERS):
    for ci, cmd in enumerate(CMD_LIST):
        for _ in range(4):
            waves.append(synth_clip(ci, si, True))
            rows.append(dict(speaker=spk, gender=GENDER[spk], dialect=DIALECT[spk],
                             text=cmd, label=COMMANDS[cmd], is_command=True))
    for _ in range(3):
        waves.append(synth_clip(0, si, False))
        rows.append(dict(speaker=spk, gender=GENDER[spk], dialect=DIALECT[spk],
                         text='(غير أمر)', label='OOS', is_command=False))

print('clips:', len(rows),
      '| commands:', sum(r['is_command'] for r in rows),
      '| out-of-set:', sum(not r['is_command'] for r in rows))

## 2. Arabic text normalization

Scoring and labels depend on a fixed normalization (Chapters 2 and 4): strip diacritics,
unify the Alef and Hamza forms, map Alef-maqsura to Ya and Ta-marbuta to Ha, and remove
the Tatweel. Publish whatever you choose, and run it on both reference and hypothesis.


In [ ]:
DIAC = re.compile(r'[\u064b-\u0652\u0670]')   # tanwin, harakat, shadda, sukun, dagger-alef
def normalize_ar(s):
    s = DIAC.sub('', s)
    s = re.sub('[\u0623\u0625\u0622]', '\u0627', s)  # hamza-alef -> bare alef
    s = s.replace('\u0649', '\u064a')                 # alef maqsura -> ya
    s = s.replace('\u0629', '\u0647')                 # ta marbuta  -> ha
    s = s.replace('\u0640', '')                        # tatweel
    return s

print('أَحْمَد ->', normalize_ar('أَحْمَد'))
for r in rows: r['text_norm'] = normalize_ar(r['text'])

## 3. Speaker-disjoint splits, with a leakage check

The single most important evaluation rule (Section 16.5): **no speaker may appear in more
than one split**. We split by speaker and assert the sets are disjoint.


In [ ]:
def speaker_splits(speakers, seed=0):
    sp = list(speakers); np.random.default_rng(seed).shuffle(sp); n = len(sp)
    return set(sp[:int(.7*n)]), set(sp[int(.7*n):int(.85*n)]), set(sp[int(.85*n):])

train_sp, dev_sp, test_sp = speaker_splits(SPEAKERS)
assert train_sp.isdisjoint(dev_sp) and train_sp.isdisjoint(test_sp) and dev_sp.isdisjoint(test_sp), 'speaker leakage!'
print('speakers  train:', len(train_sp), ' dev:', len(dev_sp), ' test:', len(test_sp), ' | disjoint OK')

## 4. Log-mel features (NumPy, following Chapter 3)

We implement the Chapter 3 front end directly: frame (25 ms, 10 ms hop), Hann window,
power spectrum, a triangular **mel filterbank** on the HTK scale, and a logarithm. We
mean-pool over time to get one 40-dimensional vector per clip for this simple baseline.


In [ ]:
def hz2mel(f): return 2595*np.log10(1 + f/700)
def mel2hz(m): return 700*(10**(m/2595) - 1)
def mel_filterbank(n_mels=40, n_fft=512, sr=SR, fmin=0, fmax=8000):
    pts = mel2hz(np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels+2))
    b = np.floor((n_fft+1)*pts/sr).astype(int)
    fb = np.zeros((n_mels, n_fft//2+1))
    for m in range(1, n_mels+1):
        l, c, r = b[m-1], b[m], b[m+1]
        for k in range(l, c):
            if c > l: fb[m-1, k] = (k-l)/(c-l)
        for k in range(c, r):
            if r > c: fb[m-1, k] = (r-k)/(r-c)
    return fb
FB = mel_filterbank()

def logmel(sig, n_fft=512, hop=160, win=400):
    w = np.hanning(win)
    frames = [sig[i:i+win]*w for i in range(0, max(1, len(sig)-win), hop)] or [np.zeros(win)]
    spec = np.abs(np.fft.rfft(np.array([np.pad(f, (0, n_fft-len(f))) for f in frames]), n=n_fft, axis=1))**2
    return np.log(FB @ spec.T + 1e-6)          # (n_mels, time)

def feature(sig): return logmel(sig).mean(axis=1)
F = np.array([feature(w) for w in waves])
print('feature matrix:', F.shape, '(clips x mel bands)')

## 5. SpecAugment (training only)

SpecAugment masks bands of time and frequency in the log-mel spectrogram (Chapter 3).
It is applied **only to training data**. Below we show it on one clip; never apply it to
the development or test sets.


In [ ]:
def spec_augment(mel, n_freq=8, n_time=10, seed=0):
    r = np.random.default_rng(seed); m = mel.copy(); n_mels, T = m.shape
    f0 = r.integers(0, max(1, n_mels-n_freq)); m[f0:f0+n_freq, :] = m.mean()
    if T > n_time:
        t0 = r.integers(0, T-n_time); m[:, t0:t0+n_time] = m.mean()
    return m

sample = logmel(waves[0])
aug = spec_augment(sample)
print('log-mel shape:', sample.shape, '| masked cells changed:', int((sample != aug).sum()))

## 6. A baseline, honestly evaluated

We train a simple classifier on the training speakers and evaluate on the **held-out**
test speakers. We include an explicit `OOS` class so the model can reject non-commands,
then report **command accuracy** (on commands) and the **out-of-set false-accept rate**
(test OOS clips predicted as a command). On this synthetic data the numbers are perfect;
on real data they will not be, which is the point of measuring them.


In [ ]:
idx = np.arange(len(rows))
def pick(split, is_cmd=None):
    return np.array([rows[i]['speaker'] in split and (is_cmd is None or rows[i]['is_command']==is_cmd) for i in idx])

tr = pick(train_sp)                       # all training clips (commands + OOS class)
Xtr, ytr = F[tr], [rows[i]['label'] for i in np.where(tr)[0]]
scaler = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=1000).fit(scaler.transform(Xtr), ytr)

cmd_te = pick(test_sp, True); oos_te = pick(test_sp, False)
acc = (clf.predict(scaler.transform(F[cmd_te])) == np.array([rows[i]['label'] for i in np.where(cmd_te)[0]])).mean()
fa  = (clf.predict(scaler.transform(F[oos_te])) != 'OOS').mean()
print(f'command accuracy (unseen speakers): {acc:.2f}')
print(f'out-of-set false-accept rate:       {fa:.2f}')

# error analysis: accuracy per command
pred = clf.predict(scaler.transform(F[cmd_te])); true = np.array([rows[i]['label'] for i in np.where(cmd_te)[0]])
for lab in sorted(set(true)):
    msk = true == lab
    print(f'  {lab:9s} accuracy {np.mean(pred[msk]==true[msk]):.2f}  (n={msk.sum()})')

## 7. The datasheet

A project's most durable output is documented data (Section 16.7; Gebru et al., 2021).
We fill the datasheet template (Appendix A) from what we know about this run.


In [ ]:
datasheet = {
  'motivation':  'Demonstrate the end-to-end lifecycle for a Gulf-dialect command recognizer (Chapter 16).',
  'composition': f'{len(rows)} clips, {len(SPEAKERS)} speakers, {len(CMD_LIST)} commands plus out-of-set, 16 kHz, synthetic.',
  'collection':  'Synthetic, deterministic audio. Replace with consented Gulf recordings or Common Voice Arabic.',
  'annotation':  'Label = command; Arabic text normalized (Section 2). Use the CODA convention for real dialect data.',
  'licensing':   'Synthetic data: free to use. For real data, state the corpus license and any no-identification clause.',
  'splits':      f'speaker-disjoint train/dev/test = {len(train_sp)}/{len(dev_sp)}/{len(test_sp)} speakers; no overlap.',
  'limitations': 'Synthetic audio is not real speech; the perfect scores check the pipeline, not real performance.',
}
for kk, vv in datasheet.items():
    print(f'{kk:12s}: {vv}')

## 8. Swapping in real data: Common Voice Arabic

To turn this demo into a real first project, replace the synthetic audio in Section 1 with
real clips and keep every other step unchanged. A common path:

```python
# !pip install datasets librosa soundfile
# from datasets import load_dataset
# cv = load_dataset('mozilla-foundation/common_voice_17_0', 'ar', split='validated', streaming=True)
# for ex in cv.take(500):
#     wav = ex['audio']['array']; sr = ex['audio']['sampling_rate']
#     # resample to 16 kHz if needed, then: waves.append(wav); rows.append({...metadata...})
```

Report the **exact Common Voice release version**, since the dataset grows with each
release, and confirm its licence terms before training or redistributing (Section 16.2).

---
**What you built:** a complete, documented Arabic speech pipeline, from data and metadata
through leak-free splits, features, a baseline, an honest evaluation, and a datasheet,
exactly the lifecycle Chapter 16 describes.
